# CS106 - Sesión 10 | Notebook 49: Modelo de Cox (Hazard Ratio) 🧬

El **Modelo de Riesgos Proporcionales de Cox** es la herramienta de modelado multivariado más potente en el análisis de supervivencia. A diferencia de las pruebas de Kaplan-Meier o Log-Rank (que solo comparan grupos de forma aislada), Cox nos permite cuantificar la magnitud exacta del riesgo y ajustar el análisis por múltiples variables o comorbilidades al mismo tiempo.

### El Hazard Ratio (HR):
Es la medida de asociación fundamental de este modelo, calculada mediante el exponencial del coeficiente, es decir: **exp(coef)**. Se interpreta comparando la tasa de riesgo instantánea de un grupo frente a un grupo de referencia basándose en tres escenarios:
* **HR = 1:** No hay diferencia de riesgo entre los grupos.
* **HR > 1:** Riesgo incrementado (ej. Un HR = 2.0 significa que ese grupo tiene el doble de velocidad o riesgo de presentar el evento en cualquier momento del tiempo).
* **HR < 1:** Efecto protector o menor tasa de riesgo (ej. Un HR = 0.50 significa la mitad del riesgo).

## 1. Ajuste del Modelo de Cox Univariado
Utilizaremos la función `coxph()` de la librería `survival`. A diferencia de los modelos lineales tradicionales (`lm`), aquí la variable dependiente obligatoria es el objeto de supervivencia `Surv(tiempo, evento)`. Evaluaremos primero el impacto aislado del **Sexo** en el riesgo de hospitalización.

In [ ]:
library(tidyverse)
library(survival)

# Cargamos el archivo de datos
datos <- read_csv("cohorte_survival.csv")

# Construimos el modelo de Cox univariado para la variable Sexo
modelo_cox_sexo <- coxph(Surv(dias_hasta_hospitalizacion, hospitalizado) ~ sexo, data = datos)

# El comando summary extrae el reporte completo con el Hazard Ratio
summary(modelo_cox_sexo)

Rows: 200 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): sexo
dbl (7): paciente_id, edad, estatura_m, peso_kg, glucosa_mgdl, hospitalizado...
lgl (3): diabetes, hipertension, obesidad

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Call:
coxph(formula = Surv(dias_hasta_hospitalizacion, hospitalizado) ~ 
    sexo, data = datos)

  n= 200, number of events= 101 

            coef exp(coef) se(coef)     z Pr(>|z|)
sexoMujer 0.2239    1.2509   0.2003 1.118    0.264

          exp(coef) exp(-coef) lower .95 upper .95
sexoMujer     1.251     0.7994    0.8448     1.852

Concordance= 0.534  (se = 0.025 )
Likelihood ratio test= 1.26  on 1 df,   p=0.3
Wald test            = 1.25  on 1 df,   p=0.3
Score (logrank) test = 1.25  on 1 df,   p=0.3


## 2. ¿Cómo leer el resultado de Cox?
Al ejecutar el resumen anterior, debes ubicarte específicamente en la columna etiquetada como **`exp(coef)`**, la cual representa el **Hazard Ratio (HR)**.

Si observas el reporte para la variable `sexoMujer`:
* El valor de `exp(coef)` ronda el **1.24**, lo que sugeriría un incremento del 24% en la velocidad de hospitalización de las mujeres frente a los hombres.
* Sin embargo, al mirar la columna del valor p (**`Pr(>|z|)`**), notarás que es mayor a 0.05 (**0.26**). Esto significa que la diferencia observada no es estadísticamente significativa y puede deberse completamente al azar.

## 3. Análisis Multivariado de Multimorbilidad
El verdadero valor del Modelo de Cox radica en evaluar la **multimorbilidad**. En la práctica médica real, un paciente no tiene una sola condición aislada; puede ser simultáneamente diabético e hipertenso, y además el riesgo se ve afectado por la edad.

**Tu misión:**
1. Ejecuta un modelo de Cox multivariado que incluya de forma simultánea los predictores `diabetes`, `hipertension` y ajuste por la variable `edad`.
2. Identifica el Hazard Ratio exacto de la **Diabetes** buscando su valor en la columna **exp(coef)** y evalúa si su impacto se mantiene estadísticamente significativo de manera independiente.
3. Responde los planteamientos clínicos basándote en la tabla impresa en la consola.

In [ ]:
# --- 1. CÓDIGO INTEGRAL DE ANÁLISIS ---
# Ajustamos el modelo multivariado con comorbilidades independientes y edad
modelo_multivariado <- coxph(Surv(dias_hasta_hospitalizacion, hospitalizado) ~ diabetes + hipertension + edad, data = datos)

# Extraemos el resumen estadístico definitivo
resumen_cox <- summary(modelo_multivariado)
print(resumen_cox)


# --- 2. SECCIÓN DE RESPUESTAS (RELLENA LOS VALORES EXACTOS OBSERVADOS) ---

# r1: ¿Cuál es el Hazard Ratio (exp(coef)) específico de la variable Diabetes?
# (Coloca el valor con 4 decimales)
r1_hr_diabetes <- 2.6727

# r2: ¿Es significativo el riesgo independiente de la diabetes (Pr(>|z|) < 0.05)?
# (Escribe TRUE o FALSE)
r2_es_significativo <- TRUE

# r3: Si un Hazard Ratio calculado en otro estudio es de 0.45, ¿el factor se considera de "Riesgo" o "Protector"?
# (Escribe exactamente "Riesgo" o "Protector")
r3_tipo_factor <- "Protector"


# --- 3. CONSTRUCCIÓN DEL DATA FRAME DE EVALUACIÓN ---
df_respuesta <- data.frame(
  item = c("hr_diabetes", "significancia_hr", "interpretacion_hr"),
  valor = c(as.character(r1_hr_diabetes),
            as.character(r2_es_significativo),
            r3_tipo_factor)
)

print(df_respuesta)

Call:
coxph(formula = Surv(dias_hasta_hospitalizacion, hospitalizado) ~ 
    diabetes + hipertension + edad, data = datos)

  n= 200, number of events= 101 

                      coef exp(coef)  se(coef)      z Pr(>|z|)    
diabetesTRUE      0.991549  2.695408  0.216949  4.570 4.87e-06 ***
hipertensionTRUE  0.687507  1.988752  0.244678  2.810  0.00496 ** 
edad             -0.001738  0.998264  0.007437 -0.234  0.81523    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

                 exp(coef) exp(-coef) lower .95 upper .95
diabetesTRUE        2.6954     0.3710    1.7618     4.124
hipertensionTRUE    1.9888     0.5028    1.2311     3.213
edad                0.9983     1.0017    0.9838     1.013

Concordance= 0.677  (se = 0.025 )
Likelihood ratio test= 38.56  on 3 df,   p=2e-08
Wald test            = 35.47  on 3 df,   p=1e-07
Score (logrank) test = 38.87  on 3 df,   p=2e-08

               item     valor
1       hr_diabetes    2.6727
2  significancia_hr      TRUE
3

In [ ]:
# Ejecuta para guardar tu avance
source("utilidades.r")
guardar_49(df_respuesta)